# Phase-2: Feature Engineering per LSOA

**Goal:** Build a master feature matrix one row per London LSOA (~4,994), one column per feature.

**Features computed:**
| Feature | Source |
|---|---|
| `crime_count` | Total crimes per LSOA (36 months) |
| `severity_weighted_count` | CCHI-weighted crime count per LSOA |
| `resolution_rate` | % crimes with positive outcome per LSOA |
| `seasonal_volatility` | Std dev of monthly crime count per LSOA |
| `imd_rank` | IMD 2025 overall deprivation rank |
| `income_rank` | IMD 2025 income deprivation rank |
| `employment_rank` | IMD 2025 employment deprivation rank |
| `stop_search_rate` | Stop & searches per km² per LSOA |
| `total_footfall` | Total TfL footfall assigned to each LSOA (36 months) |

**Note on IMD ranks:** Lower rank = more deprived. Direction is inverted in Phase 4 during normalisation.

**Output:** `outputs/phase2/phase2_feature_matrix.parquet`

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import time
from pathlib import Path
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [ ]:
# Dataset Loading
BASE   = Path('Dataset path')
P1     = BASE / 'london-final-light' / 'outputs' / 'phase1'
OUT    = BASE / 'london-final-light' / 'outputs' / 'phase2'
OUT.mkdir(parents=True, exist_ok=True)

SHP_DIR    = BASE / 'data' / 'LB_shp'
IMD_FILE   = BASE / 'data' / 'IoD-2025-custom_data_download-LSOA.csv'
GEOCACHE   = OUT / 'station_geocache.csv'

# Cambridge Crime Harm Index (CCHI 2026) severity weights for police.uk categories.
# Derived in 'severity weight/' from the official CCHI 2026 table: mean CCHI score
# (days of custody at the sentencing starting point) of the offences in each category.
# See severity weight/severity_weight_justification.md for methodology and limitations.
CCHI_WEIGHTS = {
    'Violence and sexual offences': 670,
    'Robbery':                      365,
    'Burglary':                     281,
    'Vehicle crime':                  6,
    'Theft from the person':          2,
    'Shoplifting':                    1,
    'Other theft':                    4,
    'Bicycle theft':                  5,
    'Criminal damage and arson':     98,
    'Drugs':                        156,
    'Public order':                  53,
    'Possession of weapons':        541,
    'Other crime':                   74,
    'Anti-social behaviour':          1,
}

# Positive outcome categories (resolution_rate)
RESOLVED_OUTCOMES = {
    'Suspect charged',
    'Offender given a caution',
    'Offender given a penalty notice',
    'Offender fined',
    'Offender deported',
    'Offender otherwise dealt with',
    'Suspect charged as part of another case',
    'Local resolution',
    'Offender given a drugs possession warning',
    'Offender given conditional discharge',
    'Offender given absolute discharge',
    'Offender sent to prison',
    'Offender given suspended prison sentence',
    'Offender given community sentence',
}

print('Config loaded.')
print('Output folder:', OUT)

## Section-1: Load Data

In [ ]:
print('Loading Phase 1 parquets...')
crimes      = pd.read_parquet(P1 / 'phase1_crimes_london.parquet') # put the  dataset path of your own here
outcomes    = pd.read_parquet(P1 / 'phase1_outcomes_london.parquet')
stop_search = pd.read_parquet(P1 / 'phase1_stop_search_london.parquet')
footfall    = pd.read_parquet(P1 / 'phase1_footfall_monthly.parquet')

print(f'  crimes:      {crimes.shape}')
print(f'  outcomes:    {outcomes.shape}')
print(f'  stop_search: {stop_search.shape}')
print(f'  footfall:    {footfall.shape}')

In [ ]:
print('Loading London LSOA shapefile...')
gdfs   = [gpd.read_file(f) for f in sorted(SHP_DIR.glob('*.shp'))]
london = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
london = london[['lsoa21cd', 'lsoa21nm', 'lad22nm', 'geometry']].copy()
london['area_km2'] = london.geometry.area / 1e6  # EPSG:27700 is in metres

LONDON_LSOAS = set(london['lsoa21cd'])
print(f'  Total LSOAs: {len(london)}')
print(f'  Boroughs:    {london["lad22nm"].nunique()}')
print(f'  CRS:         {london.crs}')

In [ ]:
print('Loading IMD 2025...')
imd_raw = pd.read_csv(IMD_FILE)
imd = imd_raw[[
    'LSOA code (2021)',
    'Index of Multiple Deprivation (IMD) Rank',
    'Income Rank',
    'Employment Rank',
]].copy()
imd.columns = ['lsoa21cd', 'imd_rank', 'income_rank', 'employment_rank']
print(f'  IMD shape: {imd.shape}')
print(f'  Nulls: {imd.isnull().sum().to_dict()}')

## Section-2: Filter Crimes to London LSOAs
The crime data contains records outside London (Met Police covers parts of Surrey/Hertfordshire). Filter to LSOAs present in the shapefile.

In [ ]:
before = len(crimes)
crimes_london = crimes[crimes['LSOA code'].isin(LONDON_LSOAS)].copy()
crimes_london = crimes_london.rename(columns={'LSOA code': 'lsoa21cd'})
print(f'Crimes before filter: {before:,}')
print(f'Crimes after filter:  {len(crimes_london):,}  ({before - len(crimes_london):,} outside London dropped)')
print(f'Unique London LSOAs in crime data: {crimes_london["lsoa21cd"].nunique()}')

## Section-3: Crime Count & Severity-Weighted Count

In [ ]:
# Apply CCHI weights
crimes_london['cchi_weight'] = crimes_london['Crime type'].map(CCHI_WEIGHTS).fillna(74)  # default=Other crime (CCHI 2026 mean)

crime_features = (
    crimes_london
    .groupby('lsoa21cd')
    .agg(
        crime_count=('Crime ID', 'count'),
        severity_weighted_count=('cchi_weight', 'sum'),
    )
    .reset_index()
)

print(f'Shape: {crime_features.shape}')
print(crime_features.describe().round(1))
crime_features.head(3)

## Section-4: Resolution Rate per LSOA

In [ ]:
# Join outcomes to crimes on Crime ID
# Keep only crimes with a non-null Crime ID (ASB records don't have one)
crimes_with_id = crimes_london[crimes_london['Crime ID'].notna()][['Crime ID', 'lsoa21cd']].copy()
outcomes_clean = outcomes[outcomes['Crime ID'].notna()][['Crime ID', 'Outcome type']].copy()

# Left join — crimes without an outcome record stay in (unresolved/pending)
merged = crimes_with_id.merge(outcomes_clean, on='Crime ID', how='left')
merged['resolved'] = merged['Outcome type'].apply(
    lambda x: any(r.lower() in str(x).lower() for r in RESOLVED_OUTCOMES) if pd.notna(x) else False
)

resolution = (
    merged
    .groupby('lsoa21cd')
    .agg(
        crimes_with_outcome=('Crime ID', 'count'),
        resolved_count=('resolved', 'sum'),
    )
    .reset_index()
)
resolution['resolution_rate'] = (
    resolution['resolved_count'] / resolution['crimes_with_outcome'] * 100
).round(2)

resolution = resolution[['lsoa21cd', 'resolution_rate']]
print(f'Shape: {resolution.shape}')
print(resolution['resolution_rate'].describe().round(2))
resolution.head(3)

## Section-5: Seasonal Volatility
Standard devation (Std dev) of monthly crime count per LSOA across 36 months.

In [ ]:
# Monthly crime count per LSOA
monthly_counts = (
    crimes_london
    .groupby(['lsoa21cd', 'Month'])
    .size()
    .reset_index(name='monthly_crime_count')
)

# Fill missing months with 0 for LSOAs that had no crime in that month
all_months  = crimes_london['Month'].sort_values().unique()
all_lsoas   = crimes_london['lsoa21cd'].unique()
full_index  = pd.MultiIndex.from_product([all_lsoas, all_months], names=['lsoa21cd', 'Month'])
monthly_counts = (
    monthly_counts
    .set_index(['lsoa21cd', 'Month'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

volatility = (
    monthly_counts
    .groupby('lsoa21cd')['monthly_crime_count']
    .std()
    .reset_index(name='seasonal_volatility')
)

print(f'Shape: {volatility.shape}')
print(volatility['seasonal_volatility'].describe().round(2))
volatility.head(3)

## Section-6: Stop & Search Rate (spatial join → per km²)

In [ ]:
# Filter stop_search to project period Apr 2023 – Mar 2026
ss = stop_search[
    (stop_search['Month'] >= '2023-04') &
    (stop_search['Month'] <= '2026-03')
].copy()

# Create GeoDataFrame — stop_search coords are WGS84 (EPSG:4326)
ss_gdf = gpd.GeoDataFrame(
    ss,
    geometry=gpd.points_from_xy(ss['Longitude'], ss['Latitude']),
    crs='EPSG:4326'
)
# Reproject to match London shapefile (EPSG:27700)
ss_gdf = ss_gdf.to_crs('EPSG:27700')

# Spatial join: assign each stop to an LSOA
ss_joined = gpd.sjoin(
    ss_gdf[['geometry']],
    london[['lsoa21cd', 'area_km2', 'geometry']],
    how='left',
    predicate='within'
)

# Count per LSOA and normalise by area
ss_counts = (
    ss_joined
    .dropna(subset=['lsoa21cd'])
    .groupby('lsoa21cd')
    .agg(ss_total=('lsoa21cd', 'count'), area_km2=('area_km2', 'first'))
    .reset_index()
)
ss_counts['stop_search_rate'] = (ss_counts['ss_total'] / ss_counts['area_km2']).round(4)
ss_counts = ss_counts[['lsoa21cd', 'stop_search_rate']]

matched_pct = len(ss_joined.dropna(subset=['lsoa21cd'])) / len(ss_gdf) * 100
print(f'Stop & searches matched to an LSOA: {matched_pct:.1f}%')
print(f'Shape: {ss_counts.shape}')
print(ss_counts['stop_search_rate'].describe().round(3))
ss_counts.head(3)

## Section-7: Footfall (geocode stations → spatial join → sum per LSOA)

TfL station coordinates are not in the footfall CSV. We geocode station names using Nominatim (OpenStreetMap), then spatially join to LSOAs. Results are cached to avoid re-geocoding.

In [ ]:
stations = footfall['Station'].unique()
print(f'Unique stations to geocode: {len(stations)}')

# Load cache if it exists
if GEOCACHE.exists():
    cache_df = pd.read_csv(GEOCACHE)
    geocoded = dict(zip(cache_df['station'], zip(cache_df['lat'], cache_df['lon'])))
    print(f'Loaded {len(geocoded)} cached stations')
else:
    geocoded = {}
    print('No cache found — will geocode all stations')

to_geocode = [s for s in stations if s not in geocoded]
print(f'Stations still to geocode: {len(to_geocode)}')

In [ ]:
if to_geocode:
    geolocator = Nominatim(user_agent='cbl16_project')
    geocode    = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

    print(f'Geocoding {len(to_geocode)} stations (this will take ~{len(to_geocode)//60 + 1} mins)...')
    failed = []
    for i, station in enumerate(to_geocode):
        query = f'{station} station, London, UK'
        try:
            loc = geocode(query)
            if loc:
                geocoded[station] = (loc.latitude, loc.longitude)
            else:
                # Retry with simplified query
                loc2 = geocode(f'{station}, London')
                if loc2:
                    geocoded[station] = (loc2.latitude, loc2.longitude)
                else:
                    failed.append(station)
        except Exception as e:
            failed.append(station)

        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(to_geocode)} done...')

    # Save cache
    cache_rows = [{'station': s, 'lat': v[0], 'lon': v[1]} for s, v in geocoded.items()]
    pd.DataFrame(cache_rows).to_csv(GEOCACHE, index=False)
    print(f'\nGeocoded: {len(geocoded)} | Failed: {len(failed)}')
    if failed:
        print('Failed stations:', failed)
else:
    print('All stations already cached.')

In [ ]:
# Build station GeoDataFrame
station_rows = [
    {'station': s, 'lat': v[0], 'lon': v[1]}
    for s, v in geocoded.items()
]
station_gdf = gpd.GeoDataFrame(
    pd.DataFrame(station_rows),
    geometry=gpd.points_from_xy(
        [r['lon'] for r in station_rows],
        [r['lat'] for r in station_rows]
    ),
    crs='EPSG:4326'
).to_crs('EPSG:27700')

# Spatial join: assign each station to an LSOA
station_lsoa = gpd.sjoin(
    station_gdf[['station', 'geometry']],
    london[['lsoa21cd', 'geometry']],
    how='left',
    predicate='within'
)[['station', 'lsoa21cd']]

matched_stations = station_lsoa['lsoa21cd'].notna().sum()
print(f'Stations matched to an LSOA: {matched_stations} / {len(station_gdf)}')

# Join LSOA back to footfall and sum per LSOA across all months
footfall_lsoa = (
    footfall
    .merge(station_lsoa, left_on='Station', right_on='station', how='left')
    .dropna(subset=['lsoa21cd'])
    .groupby('lsoa21cd')['TotalFootfall']
    .sum()
    .reset_index(name='total_footfall')
)

print(f'LSOAs with footfall data: {len(footfall_lsoa)}')
print(footfall_lsoa['total_footfall'].describe().round(0))
footfall_lsoa.head(3)

## Section-8: Assemble Master Feature Matrix

In [ ]:
# Start from master LSOA list (shapefile)
feature_matrix = london[['lsoa21cd', 'lsoa21nm', 'lad22nm', 'area_km2']].copy()

# Join all features
feature_matrix = feature_matrix.merge(crime_features,  on='lsoa21cd', how='left')
feature_matrix = feature_matrix.merge(resolution,      on='lsoa21cd', how='left')
feature_matrix = feature_matrix.merge(volatility,      on='lsoa21cd', how='left')
feature_matrix = feature_matrix.merge(imd,             on='lsoa21cd', how='left')
feature_matrix = feature_matrix.merge(ss_counts,       on='lsoa21cd', how='left')
feature_matrix = feature_matrix.merge(footfall_lsoa,   on='lsoa21cd', how='left')

# Fill nulls with 0 for count-based features (LSOAs with no events)
fill_zero = ['crime_count', 'severity_weighted_count', 'seasonal_volatility',
             'stop_search_rate', 'total_footfall']
feature_matrix[fill_zero] = feature_matrix[fill_zero].fillna(0)

# resolution_rate: LSOAs with no crimes get NaN (can't compute rate)
# We'll leave as NaN and handle in Phase 3

print(f'Shape: {feature_matrix.shape}')
print(f'\nNull counts:')
print(feature_matrix.isnull().sum())
feature_matrix.head(3)

In [ ]:
# Descriptive summary
numeric_cols = ['crime_count', 'severity_weighted_count', 'resolution_rate',
                'seasonal_volatility', 'imd_rank', 'income_rank',
                'employment_rank', 'stop_search_rate', 'total_footfall']
print(feature_matrix[numeric_cols].describe().round(2).to_string())

## Section-9 Save Outputs

In [ ]:
# Save feature matrix (without geometry — pure tabular)
feature_matrix.to_parquet(OUT / 'phase2_feature_matrix.parquet', index=False)
print(f'Saved phase2_feature_matrix.parquet — {feature_matrix.shape}')

# Also save monthly crime counts per LSOA (used in Phase 3 for STL/Kruskal)
monthly_counts.to_parquet(OUT / 'phase2_monthly_crime_counts.parquet', index=False)
print(f'Saved phase2_monthly_crime_counts.parquet — {monthly_counts.shape}')

print('\nPhase 2 complete.')

In [ ]:
# Phase 2 summary
print('=' * 55)
print('PHASE 2 SUMMARY')
print('=' * 55)
print(f'Total LSOAs in matrix:          {len(feature_matrix):>6}')
print(f'LSOAs with crime data:          {(feature_matrix["crime_count"] > 0).sum():>6}')
print(f'LSOAs with stop & search data:  {(feature_matrix["stop_search_rate"] > 0).sum():>6}')
print(f'LSOAs with footfall data:       {(feature_matrix["total_footfall"] > 0).sum():>6}')
print(f'LSOAs with IMD data:            {feature_matrix["imd_rank"].notna().sum():>6}')
print(f'LSOAs with resolution rate:     {feature_matrix["resolution_rate"].notna().sum():>6}')
print('=' * 55)
print(f'Outputs saved to: {OUT}')